In [3]:
import os
os.listdir('../../data/clean')

['.ipynb_checkpoints',
 'table1_uk_population.csv',
 'table3_uk_population_long.csv']

In [5]:
import pandas as pd
import sqlite3

# Correct paths AND correct filenames
df = pd.read_csv('../../data/clean/table3_uk_population_long.csv')
df_t1 = pd.read_csv('../../data/clean/table1_uk_population.csv')

# Create database
conn = sqlite3.connect('../../data/uk_population.db')

# Write tables
df.to_sql('population', conn, if_exists='replace', index=False)
df_t1.to_sql('total_population', conn, if_exists='replace', index=False)

print("Database created. Tables:")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

pop_count = pd.read_sql("SELECT COUNT(*) as rows FROM population", conn)
print(f"\nRows in population table: {pop_count['rows'][0]:,}")

Database created. Tables:
               name
0        population
1  total_population

Rows in population table: 14,580


In [6]:
q2 = """
SELECT
Year
SEX,
SUM(Population) AS Total,
ROUND(SUM(Population) / 1000000.0, 2) AS Millions
FROM population
WHERE Sex IN ('Males', 'Females')
GROUP BY Year, Sex
ORDER BY Year, Sex;
"""

df_q2 = pd.read_sql(q2, conn)
print(df_q2.tail(10).to_string())

# Correct path for notebook location
df_q2.to_csv('../../data/clean/query2_by_year.csv', index=False)

      SEX     Total  Millions
98   2020  33659504     33.66
99   2020  32492040     32.49
100  2021  33805670     33.81
101  2021  32574471     32.57
102  2022  34134567     34.13
103  2022  32891641     32.89
104  2023  34576230     34.58
105  2023  33338166     33.34
106  2024  34935953     34.94
107  2024  33720248     33.72


In [7]:
q3 = """
SELECT
YEAR,
CASE
WHEN Age BETWEEN 0 AND 17 THEN '0-17 Children'
WHEN Age BETWEEN 18 AND 64 THEN '18-64 Working age'
ELSE '65+ Older adults'
END AS Age_Group,
SUM(Population) AS Total
FROM population
WHERE Sex = 'Persons'
GROUP BY Year, Age_Group
ORDER BY Year, Age_Group;
"""

df_q3 = pd.read_sql(q3, conn)
print(df_q3.tail(15).to_string())

df_q3.to_csv('../../data/clean/query3_by_age_group_year.csv', index=False)

     Year          Age_Group     Total
147  2020      0-17 Children  13857399
148  2020  18-64 Working age  40506725
149  2020   65+ Older adults  11787420
150  2021      0-17 Children  13820535
151  2021  18-64 Working age  40613721
152  2021   65+ Older adults  11945885
153  2022      0-17 Children  13963853
154  2022  18-64 Working age  40934793
155  2022   65+ Older adults  12127562
156  2023      0-17 Children  14134503
157  2023  18-64 Working age  41462825
158  2023   65+ Older adults  12317068
159  2024      0-17 Children  14259184
160  2024  18-64 Working age  41860587
161  2024   65+ Older adults  12536430


In [8]:
q4 = """
SELECT 
Year,
SUM(Population) AS Total_Population,
LAG(SUM(Population)) OVER (ORDER BY Year) AS Prev_Year_Population,
SUM(Population) - LAG(SUM(POPULATION)) OVER (ORDER BY Year) AS YoY_Change,
ROUND(
(SUM(Population) - LAG(SUM(Population)) OVER (ORDER BY Year)) * 100.0 /
LAG(SUM(Population)) OVER (ORDER BY Year), 2
) AS YoY_Pct_Change
FROM population
WHERE Sex = 'Persons'
GROUP BY Year
ORDER BY Year;
"""

df_q4 = pd.read_sql(q4, conn)
print(df_q4.tail(10).to_string())

df_q4.to_csv('../../data/clean/query4_yoy_change.csv', index=False)

    Year  Total_Population  Prev_Year_Population  YoY_Change  YoY_Pct_Change
44  2015          64534666            64070342.0    464324.0            0.72
45  2016          65040455            64534666.0    505789.0            0.78
46  2017          65393039            65040455.0    352584.0            0.54
47  2018          65714391            65393039.0    321352.0            0.49
48  2019          66038090            65714391.0    323699.0            0.49
49  2020          66151544            66038090.0    113454.0            0.17
50  2021          66380141            66151544.0    228597.0            0.35
51  2022          67026208            66380141.0    646067.0            0.97
52  2023          67914396            67026208.0    888188.0            1.33
53  2024          68656201            67914396.0    741805.0            1.09


In [9]:
q5 = """
SELECT
p_m.Age,
p_m.Population AS Males,
p_f.Population AS Females,
ROUND(p_m.POPULATION * 100.0 / (p_m.Population + p_f.Population), 1) AS Male_Pct,
ROUND(p_f.Population * 100.0 / (p_m.Population + p_f.Population), 1) AS Female_Pct,
CASE
WHEN p_m.Population > p_f.Population THEN 'More males'
ELSE 'More females'
END AS Majority
FROM population p_m
JOIN population p_f
ON p_m.Age = p_f.Age AND p_m.Year = p_f.Year
WHERE p_m.Sex = 'Males'
AND p_f.Sex = 'Females'
AND p_m.Year = 2024
ORDER BY p_m.Age;
"""

df_q5 = pd.read_sql(q5, conn)

# Find the crossover age
crossover = df_q5[df_q5['Majority'] == 'More females']['Age'].min()
print(f"Gender crossover age (females outnumber males from age): {crossover}")

# Show a small window around the crossover
print(df_q5[df_q5['Age'].between(crossover-3, crossover+3)].to_string())

# Correct save path
df_q5.to_csv('../../data/clean/query5_gender_ratio_2024.csv', index=False)

Gender crossover age (females outnumber males from age): 26
    Age   Males  Females  Male_Pct  Female_Pct      Majority
23   23  426644   412672      50.8        49.2    More males
24   24  438356   425587      50.7        49.3    More males
25   25  448422   443801      50.3        49.7    More males
26   26  450783   454715      49.8        50.2  More females
27   27  457280   466141      49.5        50.5  More females
28   28  449418   459586      49.4        50.6  More females
29   29  451457   467367      49.1        50.9  More females


In [13]:
q6 = """
SELECT
Year,
ROUND(
SUM(Age * Population) * 1.0 / SUM(Population), 1
) AS Avg_Age
FROM population
WHERE Sex = 'Persons'
GROUP BY Year
ORDER BY Year;
"""

df_q6 = pd.read_sql(q6, conn)
print(df_q6.to_string())

df_q6.to_csv('../../data/clean/query6_avg_age_by_year.csv', index=False)

    Year  Avg_Age
0   1971     35.4
1   1972     35.4
2   1973     35.5
3   1974     35.6
4   1975     35.7
5   1976     35.9
6   1977     36.0
7   1978     36.2
8   1979     36.3
9   1980     36.4
10  1981     36.5
11  1982     36.6
12  1983     36.7
13  1984     36.9
14  1985     36.9
15  1986     37.0
16  1987     37.0
17  1988     37.1
18  1989     37.2
19  1990     37.2
20  1991     37.3
21  1992     37.3
22  1993     37.4
23  1994     37.5
24  1995     37.6
25  1996     37.7
26  1997     37.8
27  1998     37.9
28  1999     38.0
29  2000     38.1
30  2001     38.3
31  2002     38.4
32  2003     38.5
33  2004     38.5
34  2005     38.6
35  2006     38.7
36  2007     38.7
37  2008     38.8
38  2009     38.9
39  2010     39.0
40  2011     39.0
41  2012     39.1
42  2013     39.2
43  2014     39.4
44  2015     39.5
45  2016     39.6
46  2017     39.7
47  2018     39.8
48  2019     40.0
49  2020     40.2
50  2021     40.4
51  2022     40.4
52  2023     40.4
53  2024     40.4


In [14]:
q7 = """
SELECT 
Age,
Population,
RANK() OVER (ORDER BY POPULATION DESC) AS RANK
FROM population
WHERE Sex = 'Persons' AND Year = 2024
ORDER BY Population DESC
LIMIT 10;
"""

df_q7 = pd.read_sql(q7, conn)
print(df_q7.to_string())

df_q7.to_csv('../../data/clean/query7_largest_cohorts_2024.csv', index=False)

   Age  Population  RANK
0   33      976481     1
1   36      974673     2
2   34      973104     3
3   35      965382     4
4   32      965013     5
5   37      952137     6
6   38      944946     7
7   31      941624     8
8   30      940405     9
9   39      938632    10


# Notebook 03 Complete 

- All SQL queries excecuted successfully
- CSV outputs saved in 'data/clean/'
- SQL queries saved in 'sql/queries.sql'
- Database created at 'data/uk_population.db'

# Proceed to Notebook 04 for visualisation and reporting.